In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates
import numpy as np

# --- CONFIGURATION ---
GROUND_FILE = 'filtered_envisoft_air_quality_weather_data.csv'
AOD_FOLDER = 'L3_envisoft_stations'
ROOT_OUTPUT = 'Output_TimeSeries_L3'

METRICS = ['AQI', 'PM2.5', 'PM10']
METRIC_UNITS = {'AQI': 'Index', 'PM2.5': 'µg/m³', 'PM10': 'µg/m³'}
UNCERTAINTY_THRESHOLDS = [None, 1.5, 1.0, 0.5]

print("Starting L3 Time Series plotting (Corrected)...")

# --- 1. LOAD GROUND DATA ---
df_ground = pd.read_csv(GROUND_FILE, parse_dates=['Timestamp'], dayfirst=True)
df_ground['ID'] = df_ground['ID'].astype(str)

for metric in METRICS:
    if metric in df_ground.columns:
        df_ground[metric] = pd.to_numeric(df_ground[metric], errors='coerce')

station_ids = df_ground['ID'].unique()

# --- 2. PROCESSING LOOP ---
for station_id in station_ids:
    
    # Read AOD data
    aod_file_path = os.path.join(AOD_FOLDER, f"{station_id}.csv")
    if not os.path.exists(aod_file_path): continue
        
    try:
        df_aod_raw = pd.read_csv(aod_file_path)
        
        # --- CRITICAL FIX FOR L3 COLUMNS ---
        df_aod_raw = df_aod_raw.rename(columns={'timestamp': 'Timestamp'})
        
        if 'AOT_Merged' in df_aod_raw.columns:
            df_aod_raw['AOT'] = df_aod_raw['AOT_Merged']
        elif 'AOT_L2_Mean' in df_aod_raw.columns:
             df_aod_raw['AOT'] = df_aod_raw['AOT_L2_Mean']
        else:
            continue

        if 'AOT_Merged_uncertainty' in df_aod_raw.columns:
            df_aod_raw['Uncertainty'] = df_aod_raw['AOT_Merged_uncertainty']
        elif 'AOT_L2_SDV' in df_aod_raw.columns:
            df_aod_raw['Uncertainty'] = df_aod_raw['AOT_L2_SDV']
        else:
            df_aod_raw['Uncertainty'] = 0 

        df_aod_raw['Uncertainty'] = df_aod_raw['Uncertainty'].fillna(0)
        df_aod_raw['Timestamp'] = pd.to_datetime(df_aod_raw['Timestamp'])
        df_aod_raw = df_aod_raw.set_index('Timestamp')
        
    except Exception as e:
        print(f"Skipping {station_id}: {e}")
        continue

    # Prepare Ground Data
    df_station = df_ground[df_ground['ID'] == station_id].copy()
    if df_station.empty: continue
    
    # --- SAFETY FIX: SNAP GROUND DATA TO NEAREST HOUR ---
    df_station = df_station.set_index('Timestamp')
    
    # [FIX]: Added numeric_only=True so it doesn't crash on 'Name' or 'ID'
    df_station = df_station.resample('h').mean(numeric_only=True)
    
    df_station = df_station.reset_index()
    # ----------------------------------------------------
    
    # Get Name (Safe Method: Get it from the original file using ID, since df_station lost it in resampling)
    try:
        station_name = df_ground[df_ground['ID'] == station_id]['Name'].iloc[0]
    except:
        station_name = f"Station {station_id}"

    print(f"Processing ID: {station_id}")

    for metric in METRICS:
        if metric not in df_station.columns or df_station[metric].dropna().empty:
            continue

        save_dir = os.path.join(ROOT_OUTPUT, str(station_id), metric)
        os.makedirs(save_dir, exist_ok=True)

        for threshold in UNCERTAINTY_THRESHOLDS:
            
            # --- FILTER LOGIC ---
            df_filter = df_aod_raw.copy()
            df_filter = df_filter[df_filter['Uncertainty'] >= 0]
            if threshold is not None:
                df_filter = df_filter[df_filter['Uncertainty'] <= threshold]
            
            df_hourly = df_filter.resample('h').mean().dropna(subset=['AOT'])
            df_hourly = df_hourly.reset_index()
            
            merged = pd.merge(df_station, df_hourly, on='Timestamp', how='inner')
            merged = merged.sort_values('Timestamp')
            
            if len(merged) < 2: continue

            thresh_label = "ALL" if threshold is None else f"U{str(threshold).replace('.', '')}"

            # --- PLOTTING ---
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, 
                                           gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.1})
            
            # Subplot 1: Main Data
            color_metric = 'tab:blue'
            color_aot = 'tab:orange'
            
            ax1.plot(merged['Timestamp'], merged[metric], color=color_metric, 
                     marker='.', ms=4, linestyle='-', label=f'{metric} (Ground)')
            
            ax1.set_ylabel(f'{metric} ({METRIC_UNITS[metric]})', color=color_metric, fontweight='bold')
            ax1.tick_params(axis='y', labelcolor=color_metric)
            ax1.grid(True, ls='--', alpha=0.5)
            
            ax1_twin = ax1.twinx()
            ax1_twin.plot(merged['Timestamp'], merged['AOT'], color=color_aot, 
                          marker='x', ms=6, linestyle='--', label='AOT (Sat)')
            
            ax1_twin.set_ylabel('AOT', color=color_aot, fontweight='bold')
            ax1_twin.tick_params(axis='y', labelcolor=color_aot)
            
            ax1.set_title(f"Time Series: {metric} vs AOT\n{station_name}\n(Filter: {thresh_label})")
            
            lines1, labels1 = ax1.get_legend_handles_labels()
            lines2, labels2 = ax1_twin.get_legend_handles_labels()
            ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

            # Subplot 2: Uncertainty
            ax2.bar(merged['Timestamp'], merged['Uncertainty'], color='gray', alpha=0.6, width=0.04, label='Uncertainty')
            if threshold: 
                ax2.axhline(threshold, color='red', ls='--', alpha=0.8, label=f'Threshold ({threshold})')
                ax2.legend(loc='upper right')
            
            ax2.set_ylabel('Uncertainty', fontweight='bold')
            ax2.set_xlabel('Time')
            ax2.grid(True, ls='--', alpha=0.5)
            ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m\n%Hh'))
            
            filename = f"TimeSeries_{thresh_label}.png"
            plt.savefig(os.path.join(save_dir, filename), dpi=100, bbox_inches='tight')
            plt.close(fig)

print("Done. Check 'Output_TimeSeries_L3'.")

Starting L3 Time Series plotting (Corrected)...
Processing ID: 28560877461938780203765592307
Processing ID: 31390912357075263208060500522
Processing ID: 31390932574706768021562473002


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x0000028BCFA03610>>
Traceback (most recent call last):
  File "c:\Users\asiat\.conda\envs\airqua_env\lib\site-packages\ipykernel\ipkernel.py", line 797, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


Processing ID: 31390903576425084107499649578


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from scipy import stats 

# --- CONFIGURATION ---
GROUND_FILE = 'filtered_envisoft_air_quality_weather_data.csv'
AOD_FOLDER = 'L3_envisoft_stations'
ROOT_OUTPUT = 'Output_Regression_L3'

METRICS = ['AQI', 'PM2.5', 'PM10']
METRIC_UNITS = {'AQI': 'Index', 'PM2.5': 'µg/m³', 'PM10': 'µg/m³'}
UNCERTAINTY_THRESHOLDS = [None, 1.5, 1.0, 0.5]

print("Starting L3 Regression analysis...")

# --- 1. LOAD GROUND DATA ---
df_ground = pd.read_csv(GROUND_FILE, parse_dates=['Timestamp'], dayfirst=True)
df_ground['ID'] = df_ground['ID'].astype(str)

for metric in METRICS:
    if metric in df_ground.columns:
        df_ground[metric] = pd.to_numeric(df_ground[metric], errors='coerce')

station_ids = df_ground['ID'].unique()

# --- 2. PROCESSING LOOP ---
for station_id in station_ids:
    
    # Read AOD data
    aod_file_path = os.path.join(AOD_FOLDER, f"{station_id}.csv")
    if not os.path.exists(aod_file_path): continue
        
    try:
        df_aod_raw = pd.read_csv(aod_file_path)
        
        df_aod_raw = df_aod_raw.rename(columns={'timestamp': 'Timestamp'})
        
        # 1. Select AOT
        if 'AOT_Merged' in df_aod_raw.columns:
            df_aod_raw['AOT'] = df_aod_raw['AOT_Merged']
        elif 'AOT_L2_Mean' in df_aod_raw.columns:
             df_aod_raw['AOT'] = df_aod_raw['AOT_L2_Mean']
        else:
            continue 

        # 2. Select Uncertainty
        if 'AOT_Merged_uncertainty' in df_aod_raw.columns:
            df_aod_raw['Uncertainty'] = df_aod_raw['AOT_Merged_uncertainty']
        elif 'AOT_L2_SDV' in df_aod_raw.columns:
            df_aod_raw['Uncertainty'] = df_aod_raw['AOT_L2_SDV']
        else:
            df_aod_raw['Uncertainty'] = 0

        # 3. Fill NaN Uncertainty
        df_aod_raw['Uncertainty'] = df_aod_raw['Uncertainty'].fillna(0)

        # Standardize Time
        df_aod_raw['Timestamp'] = pd.to_datetime(df_aod_raw['Timestamp'])
        df_aod_raw = df_aod_raw.set_index('Timestamp')
        
    except: continue

    df_station = df_ground[df_ground['ID'] == station_id].copy()
    if df_station.empty: continue
    
    # --- SAFETY FIX: SNAP GROUND DATA TO HOURLY ---
    df_station = df_station.set_index('Timestamp')
    # ADDED numeric_only=True HERE
    df_station = df_station.resample('h').mean(numeric_only=True)
    df_station = df_station.reset_index()
    # ----------------------------------------------
    
    try:
        station_name = df_ground[df_ground['ID'] == station_id]['Name'].iloc[0]
    except:
        station_name = f"Station {station_id}"
    
    print(f"Processing Regression ID: {station_id}")

    for metric in METRICS:
        if metric not in df_station.columns or df_station[metric].dropna().empty:
            continue

        save_dir = os.path.join(ROOT_OUTPUT, str(station_id), metric)
        os.makedirs(save_dir, exist_ok=True)

        for threshold in UNCERTAINTY_THRESHOLDS:
            
            # --- FILTER LOGIC ---
            df_filter = df_aod_raw.copy()
            
            if 'Uncertainty' in df_filter.columns:
                df_filter = df_filter[df_filter['Uncertainty'] >= 0]
                if threshold is not None:
                    df_filter = df_filter[df_filter['Uncertainty'] <= threshold]
            
            df_hourly = df_filter.resample('h').mean().dropna(subset=['AOT'])
            df_hourly = df_hourly.reset_index()
            
            merged = pd.merge(df_station, df_hourly, on='Timestamp', how='inner')
            
            if len(merged) < 5: continue
            
            thresh_label = "ALL" if threshold is None else f"U{str(threshold).replace('.', '')}"

            valid_data = merged[[metric, 'AOT']].dropna()
            
            if len(valid_data) < 5: continue
            
            x = valid_data['AOT'].astype(float)
            y = valid_data[metric].astype(float)
            
            slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
            
            plt.figure(figsize=(7, 6))
            plt.scatter(x, y, alpha=0.6, edgecolors='w', s=50, label='Data points')
            
            line_x = np.array([x.min(), x.max()])
            plt.plot(line_x, slope * line_x + intercept, 'r-', lw=2, label='Fit Line')
            
            plt.title(f"Regression: {metric} vs AOT\n{station_name}\n(Filter: {thresh_label})")
            plt.xlabel("AOT (Satellite)")
            plt.ylabel(f"{metric} (Ground - {METRIC_UNITS[metric]})")
            plt.grid(True, ls='--', alpha=0.5)
            
            stats_text = '\n'.join((
                f'R = {r_value:.2f}',
                f'R² = {r_value**2:.2f}',
                f'N = {len(valid_data)}',
                f'y = {slope:.2f}x + {intercept:.2f}'
            ))
            plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
                           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
            
            filename = f"Regression_{thresh_label}.png"
            plt.savefig(os.path.join(save_dir, filename), dpi=100)
            plt.close()

print("Regression analysis complete. Check 'Output_Regression_L3'.")

Starting L3 Regression analysis...
Processing Regression ID: 28560877461938780203765592307
Processing Regression ID: 31390912357075263208060500522
Processing Regression ID: 31390932574706768021562473002
Processing Regression ID: 31390903576425084107499649578
Processing Regression ID: 31390908889087377344742439468
Processing Regression ID: 31388851800421997746903202346
Processing Regression ID: 29195707587706641566224751462
Processing Regression ID: 28505268571336961948594948504
Processing Regression ID: 31387251434693138681789561386
Processing Regression ID: 31388883344354363840031242796
Processing Regression ID: 31388839920718814259329251882
Processing Regression ID: 29213751141295132066317063859
Processing Regression ID: 31390916083317566102523755051
Regression analysis complete. Check 'Output_Regression_L3'.


In [ ]:
import pandas as pd
import os

# --- CONFIGURATION ---
AOD_FOLDER = 'L3_envisoft_stations'
OUTPUT_FILE = 'L3_Data_Diagnostics_Report.csv'

print("Starting L3 Data Forensics...")

results = []

# Get list of files
files = [f for f in os.listdir(AOD_FOLDER) if f.endswith('.csv')]

for filename in files:
    file_path = os.path.join(AOD_FOLDER, filename)
    
    try:
        df = pd.read_csv(file_path)
        
        # --- 1. Standardize Columns (Same logic as before) ---
        if 'AOT_Merged' in df.columns:
            df['AOT'] = df['AOT_Merged']
            qa_col = 'QA_flag_Merged'
        elif 'AOT_L2_Mean' in df.columns:
            df['AOT'] = df['AOT_L2_Mean']
            qa_col = 'QA_flag_Pure' # Fallback
        else:
            df['AOT'] = np.nan
            qa_col = None

        # --- 2. Calculate Statistics ---
        total_rows = len(df)
        
        # Count actual valid AOT data points
        valid_aot = df['AOT'].count()
        
        # Calculate Percentage
        completeness = (valid_aot / total_rows * 100) if total_rows > 0 else 0
        
        # --- 3. INVESTIGATE THE "MISSING" DATA ---
        # Look at the rows where AOT is NaN (Empty)
        missing_df = df[df['AOT'].isna()]
        
        top_error_reason = "Unknown"
        error_count = 0
        
        if qa_col and qa_col in df.columns and not missing_df.empty:
            # Count the most common QA flag in the missing rows
            qa_counts = missing_df[qa_col].value_counts()
            if not qa_counts.empty:
                top_error_reason = int(qa_counts.idxmax()) # Get the most frequent error code
                error_count = qa_counts.max()

        # Append to results
        results.append({
            'Station_File': filename,
            'Total_Satellite_Passes': total_rows,
            'Usable_Data_Points': valid_aot,
            'Completeness_Percentage': round(completeness, 2),
            'Primary_Missing_Reason_Code': top_error_reason,
            'Missing_Count_Due_To_Reason': error_count
        })

    except Exception as e:
        print(f"Error reading {filename}: {e}")

# --- 4. Save and Show Report ---
report_df = pd.DataFrame(results)

# Sort by lowest completeness to find the worst stations first
report_df = report_df.sort_values('Usable_Data_Points', ascending=True)

report_df.to_csv(OUTPUT_FILE, index=False)

print("-" * 60)
print(f"Diagnostic Report saved to: {OUTPUT_FILE}")
print("-" * 60)
print("WORST 5 STATIONS (Least Data):")
print(report_df[['Station_File', 'Usable_Data_Points', 'Primary_Missing_Reason_Code']].head(5))
print("-" * 60)

Starting L3 Data Forensics...
------------------------------------------------------------
Diagnostic Report saved to: L3_Data_Diagnostics_Report.csv
------------------------------------------------------------
WORST 5 STATIONS (Least Data):
                         Station_File  Usable_Data_Points  \
10  31390912357075263208060500522.csv                  31   
6   31388851800421997746903202346.csv                  43   
12  31390932574706768021562473002.csv                  45   
11  31390916083317566102523755051.csv                  51   
3   29213751141295132066317063859.csv                  59   

    Primary_Missing_Reason_Code  
10                         1533  
6                           509  
12                         1533  
11                         1533  
3                           509  
------------------------------------------------------------


In [1]:
import pandas as pd
import os
import numpy as np

# --- CONFIGURATION ---
GROUND_FILE = 'filtered_envisoft_air_quality_weather_data.csv'
AOD_FOLDER = 'L3_envisoft_stations'
OUTPUT_FILE = 'L3_Intersection_Report_Detailed.csv'

METRICS = ['AQI'] 

print("Starting Intersection Forensics (With Detailed Percentages)...")

results = []

# --- 1. LOAD GROUND DATA ---
print(f"Loading Ground Data from {GROUND_FILE}...")

try:
    df_ground = pd.read_csv(GROUND_FILE, 
                            parse_dates=['Timestamp'], 
                            dayfirst=True, 
                            dtype={'ID': str}, 
                            encoding='utf-8')
except UnicodeDecodeError:
    df_ground = pd.read_csv(GROUND_FILE, 
                            parse_dates=['Timestamp'], 
                            dayfirst=True, 
                            dtype={'ID': str}, 
                            encoding='utf-8-sig')

# Force numeric metrics
for m in METRICS:
    if m in df_ground.columns:
        df_ground[m] = pd.to_numeric(df_ground[m], errors='coerce')

station_ids = df_ground['ID'].unique()

# --- 2. LOOP STATIONS ---
for station_id in station_ids:
    
    aod_file_path = os.path.join(AOD_FOLDER, f"{station_id}.csv")
    if not os.path.exists(aod_file_path): continue

    try:
        # --- A. PREPARE SATELLITE DATA ---
        df_aod = pd.read_csv(aod_file_path)
        
        # 1. Total Raw Entries (Every time satellite passed over)
        total_sat_rows = len(df_aod)
        
        df_aod = df_aod.rename(columns={'timestamp': 'Timestamp'})
        
        if 'AOT_Merged' in df_aod.columns:
            df_aod['AOT'] = df_aod['AOT_Merged']
        elif 'AOT_L2_Mean' in df_aod.columns:
            df_aod['AOT'] = df_aod['AOT_L2_Mean']
        else:
            df_aod['AOT'] = np.nan

        df_aod['Timestamp'] = pd.to_datetime(df_aod['Timestamp'])
        df_aod = df_aod.set_index('Timestamp')
        
        # 2. Valid AOT Points (How many times it saw through clouds)
        sat_valid_mask = df_aod['AOT'].notna()
        count_sat_valid = sat_valid_mask.sum()
        
        # --- B. PREPARE GROUND DATA ---
        df_station = df_ground[df_ground['ID'] == station_id].copy()
        
        try:
             name = df_station['Name'].dropna().iloc[0]
        except: name = "Unknown"

        # Snap to hourly
        df_station = df_station.set_index('Timestamp')
        df_station = df_station.resample('h').mean(numeric_only=True)
        
        ground_valid_mask = df_station['AQI'].notna()
        count_ground_valid = ground_valid_mask.sum()

        # --- C. FIND THE INTERSECTION ---
        valid_sat_df = df_aod[sat_valid_mask].reset_index()[['Timestamp']]
        valid_ground_df = df_station[ground_valid_mask].reset_index()[['Timestamp']]
        
        intersection = pd.merge(valid_sat_df, valid_ground_df, on='Timestamp', how='inner')
        count_intersection = len(intersection)
        
        # --- D. CALCULATE PERCENTAGES ---
        
        # Percentage 1: Cloud/Algorithm Success Rate
        # (Valid AOT / Total Passes)
        if total_sat_rows > 0:
            sat_usability_pct = round((count_sat_valid / total_sat_rows) * 100, 2)
        else:
            sat_usability_pct = 0
            
        # Percentage 2: Ground Station Overlap Rate
        # (Matches / Valid AOT) - "Did the station work when the satellite worked?"
        if count_sat_valid > 0:
            match_success_pct = round((count_intersection / count_sat_valid) * 100, 2)
        else:
            match_success_pct = 0

        results.append({
            'Station_ID': str(station_id),
            'Station_Name': name,
            'Total_Sat_Rows': total_sat_rows,
            'Sat_Valid_Points': count_sat_valid,
            'MATCHES': count_intersection,
            'Sat_Usability_%': sat_usability_pct, # How often satellite sees ground
            'Match_Success_%': match_success_pct  # How often ground matches satellite
        })

    except Exception as e:
        print(f"Error processing {station_id}: {e}")

# --- 3. SAVE REPORT ---
report_df = pd.DataFrame(results)
# Sort by Matches (Lowest first) to find the problem stations
report_df = report_df.sort_values('MATCHES', ascending=True)

report_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print("-" * 60)
print(f"Report saved to: {OUTPUT_FILE}")
print("-" * 60)
print("PREVIEW (Lowest Matches):")
if not report_df.empty:
    cols = ['Station_Name', 'Total_Sat_Rows', 'Sat_Valid_Points', 'Sat_Usability_%', 'Match_Success_%', 'MATCHES']
    print(report_df[cols].head(10).to_string())
else:
    print("No matches found.")

Starting Intersection Forensics (With Detailed Percentages)...
Loading Ground Data from filtered_envisoft_air_quality_weather_data.csv...
------------------------------------------------------------
Report saved to: L3_Intersection_Report_Detailed.csv
------------------------------------------------------------
PREVIEW (Lowest Matches):
                                                                                     Station_Name  Total_Sat_Rows  Sat_Valid_Points  Sat_Usability_%  Match_Success_%  MATCHES
1   HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận 2 (Ngã ba Lê Hữu Kiểu và Trương Văn Bang) (KK)            8728                31             0.36             6.45        2
6                                             Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK)            8728               105             1.20             1.90        2
5                                              Đà Nẵng: Khuôn viên trường ĐH sư phạm Đà Nẵng (KK)            8728                43     

In [ ]:
import pandas as pd
import os
import numpy as np

# --- CONFIGURATION ---
GROUND_FILE = 'filtered_envisoft_air_quality_weather_data.csv'
AOD_FOLDER = 'L3_envisoft_stations'
OUTPUT_FILE = 'L3_Hourly_Missed_Log.csv'

print("Starting HOURLY Forensics...")

# --- 1. LOAD GROUND DATA ---
df_ground = pd.read_csv(GROUND_FILE, parse_dates=['Timestamp'], dayfirst=True)
df_ground['ID'] = df_ground['ID'].astype(str)
if 'AQI' in df_ground.columns:
    df_ground['AQI'] = pd.to_numeric(df_ground['AQI'], errors='coerce')

station_ids = df_ground['ID'].unique()
hourly_misses = []

# --- 2. LOOP THROUGH STATIONS ---
for station_id in station_ids:
    
    aod_file_path = os.path.join(AOD_FOLDER, f"{station_id}.csv")
    if not os.path.exists(aod_file_path): continue

    try:
        # --- PREPARE SATELLITE (HOURLY) ---
        df_aod = pd.read_csv(aod_file_path)
        df_aod = df_aod.rename(columns={'timestamp': 'Timestamp'})
        
        # Select Best AOT
        if 'AOT_Merged' in df_aod.columns:
            df_aod['AOT'] = df_aod['AOT_Merged']
        elif 'AOT_L2_Mean' in df_aod.columns:
             df_aod['AOT'] = df_aod['AOT_L2_Mean']
        else: continue

        df_aod['Timestamp'] = pd.to_datetime(df_aod['Timestamp'])
        df_aod = df_aod.set_index('Timestamp')
        
        # Keep Hourly Resolution
        df_aod_hourly = df_aod.resample('h').mean().dropna(subset=['AOT'])
        
        # --- PREPARE GROUND (HOURLY) ---
        df_station = df_ground[df_ground['ID'] == station_id].copy()
        
        # Get Name
        try: station_name = df_station['Name'].iloc[0]
        except: station_name = f"Station {station_id}"
        
        df_station = df_station.set_index('Timestamp')
        
        # Snap Ground to Hourly
        df_station_hourly = df_station.resample('h').mean(numeric_only=True)
        
        # --- FIND MISSING DATA (HOURLY) ---
        # Left Join: Keep Satellite hours, check Ground
        merged = pd.merge(df_aod_hourly, df_station_hourly, on='Timestamp', how='left')
        
        # Find rows where AOT exists but AQI is Missing
        misses = merged[merged['AQI'].isna()].copy()
        
        for timestamp, row in misses.iterrows():
            hourly_misses.append({
                'Station_Name': station_name,
                'Station_ID': station_id,
                'Exact_Time_Missed': timestamp, 
                'Satellite_AOT': round(row['AOT'], 3),
                'Ground_AQI': 'MISSING'
            })

    except Exception as e:
        print(f"Skipping {station_id}: {e}")

# --- 3. SAVE ---
df_report = pd.DataFrame(hourly_misses)

if not df_report.empty:
    df_report = df_report.sort_values(by=['Station_Name', 'Exact_Time_Missed'])
    df_report.to_csv(OUTPUT_FILE, index=True) 
    
    print("-" * 60)
    print(f"Hourly Log saved to: {OUTPUT_FILE}")
    print(f"Found {len(df_report)} specific HOURS where data was lost.")
    print("-" * 60)
    print("PREVIEW:")
    print(df_report[['Station_Name', 'Exact_Time_Missed', 'Satellite_AOT']].head(5))
else:
    print("No hourly mismatches found.")

Starting HOURLY Forensics...
------------------------------------------------------------
Hourly Log saved to: L3_Hourly_Missed_Log.csv
Found 919 specific HOURS where data was lost.
------------------------------------------------------------
PREVIEW:
                                          Station_Name   Exact_Time_Missed  \
773  Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ... 2025-01-07 09:00:00   
774  Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ... 2025-01-07 10:00:00   
775  Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ... 2025-01-07 11:00:00   
776  Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ... 2025-01-09 11:00:00   
777  Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ... 2025-01-13 09:00:00   

     Satellite_AOT  
773          0.437  
774          0.375  
775          0.421  
776          0.324  
777          0.148  


In [8]:
import pandas as pd
import os
import numpy as np

# --- CONFIGURATION ---
GROUND_FILE = 'filtered_envisoft_air_quality_weather_data.csv'
AOD_FOLDER = 'L3_envisoft_stations'
OUTPUT_FILE = 'L3_Matched_Data_Points.csv'

print("Starting Match Extraction...")

all_matches = []

# --- 1. LOAD GROUND DATA ---
print(f"Loading Ground Data from {GROUND_FILE}...")
try:
    df_ground = pd.read_csv(GROUND_FILE, 
                            parse_dates=['Timestamp'], 
                            dayfirst=True, 
                            dtype={'ID': str}, 
                            encoding='utf-8')
except UnicodeDecodeError:
    df_ground = pd.read_csv(GROUND_FILE, 
                            parse_dates=['Timestamp'], 
                            dayfirst=True, 
                            dtype={'ID': str}, 
                            encoding='utf-8-sig')

# Force numeric
if 'AQI' in df_ground.columns:
    df_ground['AQI'] = pd.to_numeric(df_ground['AQI'], errors='coerce')

station_ids = df_ground['ID'].unique()

# --- 2. LOOP STATIONS ---
for station_id in station_ids:
    
    aod_file_path = os.path.join(AOD_FOLDER, f"{station_id}.csv")
    if not os.path.exists(aod_file_path): continue

    try:
        # --- A. PREPARE SATELLITE DATA ---
        df_aod = pd.read_csv(aod_file_path)
        df_aod = df_aod.rename(columns={'timestamp': 'Timestamp'})
        
        # 1. Select Best AOT
        if 'AOT_Merged' in df_aod.columns:
            df_aod['AOT'] = df_aod['AOT_Merged']
        elif 'AOT_L2_Mean' in df_aod.columns:
            df_aod['AOT'] = df_aod['AOT_L2_Mean']
        else: continue

        # 2. Select Uncertainty (The column you asked for)
        if 'AOT_Merged_uncertainty' in df_aod.columns:
            df_aod['Uncertainty'] = df_aod['AOT_Merged_uncertainty']
        elif 'AOT_L2_SDV' in df_aod.columns:
            df_aod['Uncertainty'] = df_aod['AOT_L2_SDV']
        else:
            df_aod['Uncertainty'] = np.nan # Mark as unknown if missing

        df_aod['Timestamp'] = pd.to_datetime(df_aod['Timestamp'])
        df_aod = df_aod.set_index('Timestamp')
        
        # Filter: Only keep rows where we actually have AOT
        df_aod_valid = df_aod.dropna(subset=['AOT'])
        
        # --- B. PREPARE GROUND DATA ---
        df_station = df_ground[df_ground['ID'] == station_id].copy()
        
        # Get Name
        try: name = df_station['Name'].dropna().iloc[0]
        except: name = f"Station {station_id}"

        # Snap to hourly
        df_station = df_station.set_index('Timestamp')
        df_station = df_station.resample('h').mean(numeric_only=True)
        
        # Filter: Only keep rows where we have AQI
        df_station_valid = df_station.dropna(subset=['AQI'])

        # --- C. MERGE (THE INTERSECTION) ---
        # Inner join to find the matches
        merged = pd.merge(df_aod_valid[['AOT', 'Uncertainty']], 
                          df_station_valid[['AQI']], 
                          left_index=True, 
                          right_index=True, 
                          how='inner')
        
        if not merged.empty:
            merged = merged.reset_index()
            merged['Station_ID'] = str(station_id)
            merged['Station_Name'] = name
            
            # Reorder columns for readability
            cols = ['Station_ID', 'Station_Name', 'Timestamp', 'AQI', 'AOT', 'Uncertainty']
            all_matches.append(merged[cols])

    except Exception as e:
        print(f"Error processing {station_id}: {e}")

# --- 3. SAVE RESULTS ---
if all_matches:
    final_df = pd.concat(all_matches, ignore_index=True)
    
    # Sort by Station Name then Time
    final_df = final_df.sort_values(by=['Station_Name', 'Timestamp'])
    
    final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    
    print("-" * 60)
    print(f"Saved {len(final_df)} data points to: {OUTPUT_FILE}")
    print("-" * 60)
    print("PREVIEW OF THE SURVIVORS:")
    print(final_df[['Station_Name', 'Timestamp', 'AQI', 'AOT', 'Uncertainty']].head(10).to_string())
else:
    print("No matches found across any stations.")

Starting Match Extraction...
Loading Ground Data from filtered_envisoft_air_quality_weather_data.csv...
------------------------------------------------------------
Saved 320 data points to: L3_Matched_Data_Points.csv
------------------------------------------------------------
PREVIEW OF THE SURVIVORS:
                                                                       Station_Name           Timestamp    AQI     AOT  Uncertainty
292                        Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK) 2025-05-02 09:00:00   37.0  0.1690          NaN
293                        Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK) 2025-11-26 09:00:00   72.0  0.1416          NaN
294                        Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK) 2025-11-26 10:00:00   66.0  0.1166          NaN
295                        Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp Thành (KK) 2025-11-26 11:00:00   61.0  0.1892       0.6455
296                        Bình Dươ

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.linear_model import RANSACRegressor
from sklearn.metrics import r2_score
import shutil  # Added for folder cleanup

# --- CONFIGURATION ---
GROUND_FILE = 'filtered_envisoft_air_quality_weather_data.csv'
AOD_FOLDER = 'L3_envisoft_stations'
ROOT_OUTPUT = 'L3_Envisoft_RANSAC_Output'

METRICS = ['AQI'] 
UNCERTAINTY_THRESHOLDS = [None, 0.5] 

print("Starting Organized RANSAC Analysis...")

# --- 1. LOAD GROUND DATA ---
try:
    df_ground = pd.read_csv(GROUND_FILE, parse_dates=['Timestamp'], dayfirst=True, dtype={'ID': str}, encoding='utf-8')
except:
    df_ground = pd.read_csv(GROUND_FILE, parse_dates=['Timestamp'], dayfirst=True, dtype={'ID': str}, encoding='utf-8-sig')

for metric in METRICS:
    if metric in df_ground.columns:
        df_ground[metric] = pd.to_numeric(df_ground[metric], errors='coerce')

station_ids = df_ground['ID'].unique()

# --- 2. PROCESSING LOOP ---
for station_id in station_ids:
    
    aod_file_path = os.path.join(AOD_FOLDER, f"{station_id}.csv")
    if not os.path.exists(aod_file_path): continue
        
    try: 
        # --- A. PREPARE SATELLITE DATA ---
        df_aod_raw = pd.read_csv(aod_file_path)
        df_aod_raw = df_aod_raw.rename(columns={'timestamp': 'Timestamp'})
        
        if 'AOT_Merged' in df_aod_raw.columns:
            df_aod_raw['AOT'] = df_aod_raw['AOT_Merged']
        elif 'AOT_L2_Mean' in df_aod_raw.columns:
             df_aod_raw['AOT'] = df_aod_raw['AOT_L2_Mean']
        else: continue 

        if 'AOT_Merged_uncertainty' in df_aod_raw.columns:
            df_aod_raw['Uncertainty'] = df_aod_raw['AOT_Merged_uncertainty']
        elif 'AOT_L2_SDV' in df_aod_raw.columns:
            df_aod_raw['Uncertainty'] = df_aod_raw['AOT_L2_SDV']
        else: df_aod_raw['Uncertainty'] = 0

        df_aod_raw['Uncertainty'] = df_aod_raw['Uncertainty'].fillna(0)
        df_aod_raw['Timestamp'] = pd.to_datetime(df_aod_raw['Timestamp'])
        df_aod_raw = df_aod_raw.set_index('Timestamp')

        # --- B. PREPARE GROUND DATA ---
        df_station = df_ground[df_ground['ID'] == station_id].copy()
        if df_station.empty: continue
        
        try: 
            full_name = df_station['Name'].iloc[0]
            if ':' in full_name:
                city_name = full_name.split(':')[0].strip()
                station_name = full_name.split(':')[1].strip()
            else:
                city_name = "Other_Provinces"
                station_name = full_name
        except: 
            city_name = "Unknown"
            station_name = f"Station_{station_id}"
        
        safe_city = "".join([c for c in city_name if c.isalnum() or c in (' ', '-', '_')]).strip()
        safe_station = "".join([c for c in station_name if c.isalnum() or c in (' ', '-', '_')]).strip()

        # --- FLAT-LINE FILTER ---
        df_station = df_station.sort_values('Timestamp')
        for m in METRICS:
            df_station[f'{m}_diff'] = df_station[m].diff()
            df_station.loc[df_station[f'{m}_diff'] == 0, m] = np.nan

        df_station = df_station.set_index('Timestamp')
        df_station = df_station.resample('h').mean(numeric_only=True)
        df_station = df_station.reset_index()

        # --- THRESHOLD FILTER (AQI >= 50) ---
        for m in METRICS:
            df_station = df_station[df_station[m] >= 50]

        # --- C. MERGE & REGRESS ---
        for metric in METRICS:
            if df_station[metric].dropna().empty: continue

            # Pre-create directory (we will delete it later if it stays empty)
            save_dir = os.path.join(ROOT_OUTPUT, safe_city)
            os.makedirs(save_dir, exist_ok=True)

            for threshold in UNCERTAINTY_THRESHOLDS:
                df_filter = df_aod_raw.copy()
                df_filter = df_filter[df_filter['Uncertainty'] >= 0]
                if threshold is not None:
                    df_filter = df_filter[df_filter['Uncertainty'] <= threshold]
                
                df_hourly = df_filter.resample('h').mean().dropna(subset=['AOT'])
                df_hourly = df_hourly.reset_index()
                
                merged = pd.merge(df_station, df_hourly, on='Timestamp', how='inner')
                if len(merged) < 15: continue
                
                thresh_label = "ALL" if threshold is None else f"U{str(threshold).replace('.', '')}"
                valid_data = merged[[metric, 'AOT']].dropna()
                
                X = valid_data['AOT'].values.reshape(-1, 1)
                y = valid_data[metric].values

                target_count = int(0.8 * len(X))
                if target_count < 10: target_count = 10

                try:
                    # RANSAC (With 80% Inlier Target)
                    ransac = RANSACRegressor(random_state=42, min_samples=10, stop_n_inliers=target_count, residual_threshold=None)
                    ransac.fit(X, y)
                    
                    inlier_mask = ransac.inlier_mask_
                    outlier_mask = np.logical_not(inlier_mask)
                    r2 = r2_score(y[inlier_mask], ransac.predict(X[inlier_mask]))
                    slope = ransac.estimator_.coef_[0]
                    intercept = ransac.estimator_.intercept_

                    # Plotting
                    plt.figure(figsize=(8, 7))
                    plt.scatter(X[outlier_mask], y[outlier_mask], color='red', marker='x', alpha=0.5, label='Outliers')
                    plt.scatter(X[inlier_mask], y[inlier_mask], color='blue', marker='o', alpha=0.7, label='Inliers')
                    
                    line_X = np.arange(X.min(), X.max(), 0.01)[:, np.newaxis]
                    line_y = ransac.predict(line_X)
                    plt.plot(line_X, line_y, color='green', linewidth=2, label='RANSAC Fit')
                    
                    plt.title(f"{city_name} - {station_name}\n{metric} vs AOT (Filter: {thresh_label})")
                    plt.xlabel("AOT (Satellite)")
                    plt.ylabel(f"{metric} (Ground)")
                    plt.grid(True, ls='--', alpha=0.5)
                    plt.legend()
                    
                    stats_text = '\n'.join((
                        f'RANSAC R² = {r2:.2f}',
                        f'Inliers = {np.sum(inlier_mask)}/{len(y)}',
                        f'y = {slope:.2f}x + {intercept:.2f}'
                    ))
                    plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
                                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
                    
                    filename = f"{safe_city}_{safe_station}_{metric}_{thresh_label}_RANSAC.png"
                    plt.savefig(os.path.join(save_dir, filename), dpi=100)
                    plt.close()
                    print(f"Saved: {safe_city}/{filename}")

                except Exception as e:
                    pass

    except Exception as e: 
        print(f"Error processing {station_id}: {e}")

# --- 3. CLEANUP: REMOVE EMPTY FOLDERS ---
print("Cleaning up empty folders...")
if os.path.exists(ROOT_OUTPUT):
    for city_folder in os.listdir(ROOT_OUTPUT):
        folder_path = os.path.join(ROOT_OUTPUT, city_folder)
        if os.path.isdir(folder_path):
            # Check if directory is empty
            if not os.listdir(folder_path):
                os.rmdir(folder_path)
                print(f"Removed empty folder: {city_folder}")

print(f"Analysis Complete. Check folder: {ROOT_OUTPUT}")

Starting Organized RANSAC Analysis...
Saved: Hà Nội/Hà Nội_556 Nguyễn Văn Cừ KK_AQI_ALL_RANSAC.png
Saved: Hà Nội/Hà Nội_556 Nguyễn Văn Cừ KK_AQI_U05_RANSAC.png
Saved: Hà Nội/Hà Nội_ĐHBK cổng Parabol đường Giải Phóng KK_AQI_ALL_RANSAC.png
Saved: Hà Nội/Hà Nội_ĐHBK cổng Parabol đường Giải Phóng KK_AQI_U05_RANSAC.png
Saved: Hà Nội/Hà Nội_Công viên Nhân Chính - Khuất Duy Tiến KK_AQI_ALL_RANSAC.png
Saved: Hà Nội/Hà Nội_Công viên Nhân Chính - Khuất Duy Tiến KK_AQI_U05_RANSAC.png
Saved: Bắc Giang/Bắc Giang_Khu liên cơ quan tỉnh Bắc Giang - P Ngô Quyền - TP Bắc Giang KK_AQI_ALL_RANSAC.png
Saved: Bắc Giang/Bắc Giang_Khu liên cơ quan tỉnh Bắc Giang - P Ngô Quyền - TP Bắc Giang KK_AQI_U05_RANSAC.png
Saved: Hà Nam/Hà Nam_Công Viên Nam Cao - PQuang Trung - TP Phủ Lý KK_AQI_ALL_RANSAC.png
Saved: Hà Nam/Hà Nam_Công Viên Nam Cao - PQuang Trung - TP Phủ Lý KK_AQI_U05_RANSAC.png
Cleaning up empty folders...
Removed empty folder: Bình Dương
Removed empty folder: HCM
Removed empty folder: Long An
Removed 